In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rudrakumargupta/banking-faq-dataset-for-chatbot-training/banking_knowledge_base_1000.csv


In [2]:
df=pd.read_csv("/kaggle/input/datasets/rudrakumargupta/banking-faq-dataset-for-chatbot-training/banking_knowledge_base_1000.csv")

In [3]:
df.head(5)

,Section,Question,Answer
0,Basic Accounts,What is a Savings Account?,A savings account lets you keep your money saf...
1,Basic Accounts,What is a Current Account?,A current account is mainly for businesses. It...
2,Basic Accounts,What is a Fixed Deposit?,A fixed deposit lets you lock in a sum of mone...
3,Basic Accounts,What does KYC stand for?,KYC stands for Know Your Customer. It's how ba...
4,Basic Accounts,What is a cheque?,A cheque is a written instruction to your bank...


In [4]:
df.tail(5)

,Section,Question,Answer
995,Insurance & Govt Schemes,What is the PM Garib Kalyan Anna Yojana?,PMGKAY provides free food grain (5 kg per pers...
996,International Banking,What is the RBI's LRS limit?,"Under the Liberalised Remittance Scheme, resid..."
997,Financial Markets,What is a balanced advantage fund?,A balanced advantage fund (BAF) dynamically sh...
998,Banking Terminology,What is a bank's interest coverage ratio?,Interest coverage ratio measures how easily a ...
999,Basic Accounts,What is e-statement in banking?,An e-statement is a digital version of your mo...


In [5]:
df["Section"].nunique

<bound method IndexOpsMixin.nunique of 0                Basic Accounts
1                Basic Accounts
2                Basic Accounts
3                Basic Accounts
4                Basic Accounts
                 ...           
995    Insurance & Govt Schemes
996       International Banking
997           Financial Markets
998         Banking Terminology
999              Basic Accounts
Name: Section, Length: 1000, dtype: object>

In [6]:
df.isnull().sum()

Section     0
Question    0
Answer      0
dtype: int64

In [7]:
Answer_df = df.explode("Answer", ignore_index=True)
Answer_df.head(4)

,Section,Question,Answer
0,Basic Accounts,What is a Savings Account?,A savings account lets you keep your money saf...
1,Basic Accounts,What is a Current Account?,A current account is mainly for businesses. It...
2,Basic Accounts,What is a Fixed Deposit?,A fixed deposit lets you lock in a sum of mone...
3,Basic Accounts,What does KYC stand for?,KYC stands for Know Your Customer. It's how ba...


In [8]:
from datasets import Dataset

Answer_dataset = Dataset.from_pandas(Answer_df)
Answer_dataset = Answer_dataset.map(
    lambda x: {"Answer_length": len(x["Answer"].split())}
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
Answer_dataset = Answer_dataset.filter(lambda x: x["Answer_length"] > 15)
Answer_dataset

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['Section', 'Question', 'Answer', 'Answer_length'],
    num_rows: 999
})

In [10]:
def concatenate_text(examples):
    return {
        "text": examples["Section"]
        + " \n "
        + examples["Question"]
        + " \n "
        + examples["Answer"]
    }


Answer_dataset = Answer_dataset.map(concatenate_text)

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

In [11]:
Answer_dataset[1]

{'Section': 'Basic Accounts',
 'Question': 'What is a Current Account?',
 'Answer': "A current account is mainly for businesses. It allows unlimited daily transactions but doesn't pay interest on the balance.",
 'Answer_length': 19,
 'text': "Basic Accounts \n What is a Current Account? \n A current account is mainly for businesses. It allows unlimited daily transactions but doesn't pay interest on the balance."}

In [12]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
import torch

device = torch.device("cuda")
model.to(device)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [14]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

In [15]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [16]:
embedding = get_embeddings(Answer_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

In [17]:
embeddings_dataset = Answer_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

In [18]:
!pip uninstall -y numpy faiss faiss-cpu
!pip install numpy==1.26.4
!pip install faiss-cpu

Found existing installation: numpy 2.4.6
Uninstalling numpy-2.4.6:
  Successfully uninstalled numpy-2.4.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 929.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 13.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.

In [19]:
import importlib
import datasets.search

importlib.reload(datasets.search)

<module 'datasets.search' from '/usr/local/lib/python3.12/dist-packages/datasets/search.py'>

In [20]:
import faiss

index = faiss.IndexFlatL2(768)
embeddings_dataset.add_faiss_index(column="embeddings")

  0%|          | 0/1 [00:00<?, ?it/s]

Dataset({
    features: ['Section', 'Question', 'Answer', 'Answer_length', 'text', 'embeddings'],
    num_rows: 999
})

In [21]:
question = "My debit card got stolen, how can I block it?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

(1, 768)

In [22]:
scores, samples = embeddings_dataset.get_nearest_examples("embeddings",question_embedding,k=5)

In [23]:
print(scores)

[21.361874 29.066975 33.37573  33.459915 33.846344]


In [24]:
type(samples)

dict

In [25]:
import pandas as pd

samples_df = pd.DataFrame(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)
#samples_df.tail(1)["Answer"]
print(samples_df.iloc[-1]["Answer"])

Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


In [26]:
print(question)
print(samples_df.iloc[-1]["Answer"])
print(samples_df.tail(1)["scores"])

My debit card got stolen, how can I block it?
Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.
0    21.361874
Name: scores, dtype: float32


In [27]:
print(samples_df[["Question","scores"]])

                                        Question     scores
4           How do I report fraud in my account?  33.846344
3          How do I set a PIN for my debit card?  33.459915
2  How do I report a lost or stolen credit card?  33.375729
1                  What is a card block request?  29.066975
0                  How do I block my debit card?  21.361874
